In [14]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

# =====================================================================
# 1. DEFINICJA STRUKTURY MODELU (musi być identyczna jak przy treningu)
# =====================================================================
class OptimizedHouseholdLSTM(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout
        )
        self.fc = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

# =====================================================================
# 2. WCZYTANIE MODELU I SCALERÓW Z CHECKPOINTU
# =====================================================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MODEL_PATH = 'household_lstm_v2_optimized.pth'

# Wczytujemy cały zapisany słownik
checkpoint = torch.load(MODEL_PATH, map_location=device, weights_only=False)

# Odtwarzamy model na podstawie parametrów z checkpointu
model = OptimizedHouseholdLSTM(
    input_size=checkpoint['input_size'],
    hidden_size=checkpoint['hidden_size'],
    num_layers=checkpoint['num_layers'],
    dropout=checkpoint['dropout']
).to(device)

model.load_state_dict(checkpoint['model_state_dict'])
model.eval()  # Przełączenie w tryb predykcji (wyłącza dropout)

# Pobieramy dopasowane wcześniej scalery i parametry
scaler_X = checkpoint['scaler_X']
scaler_y = checkpoint['scaler_y']
feature_cols = checkpoint['feature_cols']
WINDOW_SIZE = checkpoint['window_size']

print(f"Model wczytany pomyślnie na urządzeniu: {device}")
print(f"Liczba cech: {len(feature_cols)}, Długość okna (WINDOW_SIZE): {WINDOW_SIZE}")

# =====================================================================
# 3. PRZYGOTOWANIE NOWYCH DANYCH I PREDYKCJA
# =====================================================================
def prepare_data_and_predict(df_raw, model, scaler_X, scaler_y, feature_cols, window_size):
    df = df_raw.copy()
    
    # 1. Usunięcie niepotrzebnych kolumn (jeśli istnieją w nowych danych)
    dead_and_noisy_cols = [
        'wifi_router_kwh', 'standby_devices_kwh', 'pet_feeder_kwh',
        'pet_camera_kwh', 'hour', 'apparent_temperature'
    ]
    df = df.drop(columns=[col for col in dead_and_noisy_cols if col in df.columns])

    # 2. Tworzenie cech autoregresyjnych (lags i średnie kroczące)
    if 'lag_1' not in df.columns:
        df['lag_1'] = df['total_kwh'].shift(1)
        df['lag_2'] = df['total_kwh'].shift(2)
        df['lag_24'] = df['total_kwh'].shift(24)
        df['lag_168'] = df['total_kwh'].shift(168)
        df['rolling_mean_6h'] = df['total_kwh'].shift(1).rolling(6).mean()
        df['rolling_mean_24h'] = df['total_kwh'].shift(1).rolling(24).mean()

    # Usunięcie NaN po shift/rolling
    df = df.dropna().reset_index(drop=True)

    # Upewniamy się, że kolumny są w identycznej kolejności
    X_raw = df[feature_cols]

    # 3. Skalowanie cech
    X_scaled = scaler_X.transform(X_raw)

    # 4. Tworzenie sekwencji do LSTM
    X_seq = []
    for i in range(len(X_scaled) - window_size):
        X_seq.append(X_scaled[i : i + window_size])
    
    X_seq = np.array(X_seq)
    
    if len(X_seq) == 0:
        raise ValueError(f"Za mało danych! Wymagane minimum to {window_size + 1} wierszy po czyszczeniu.")

    # 5. Konwersja na Tensor i predykcja
    X_tensor = torch.tensor(X_seq, dtype=torch.float32).to(device)
    
    with torch.no_grad():
        preds_scaled = model(X_tensor).cpu().numpy()

    # 6. Odwrócenie skalowania do oryginalnych kWh
    preds_kwh = scaler_y.inverse_transform(preds_scaled)

    # Pobieramy prawdziwe wartości total_kwh dopasowane czasowo do predykcji
    actual_kwh = df['total_kwh'].values[window_size:]

    results_df = pd.DataFrame({
        'Rzeczywiste_kWh': actual_kwh,
        'Predykcja_kWh': preds_kwh.flatten(),
        'Różnica_kWh': actual_kwh - preds_kwh.flatten()
    })

    return results_df

# =====================================================================
# 4. URUCHOMIENIE PREDYKCJI DLA NOWEGO PLIKU LUB PRÓBKI
# =====================================================================
# Podmień 'new_household_data.csv' na ścieżkę do Twojego pliku z danymi
new_df = pd.read_csv('household_full_profile.csv') 

results = prepare_data_and_predict(new_df, model, scaler_X, scaler_y, feature_cols, WINDOW_SIZE)

# Wyświetlenie pierwszych 10 predykcji
print("\n" + "=" * 50)
print("  PIERWSZE 10 PREDYKCJI MODELU (kWh)")
print("=" * 50)
print(results.head(10).to_string(index=False))

ModuleNotFoundError: No module named 'sklearn'